# 7) Voltage and reactive power — Exercises

**Goals:** compute surge impedance loadings, find how light a night must be to cause high voltages, compare a fixed with a variable reactor, size a must-run plant's reactive capability, and test whether switching lines off helps.

The setup cell holds the lecture's `network(load_scale, plants, wind)`, `night(load_scale)` (no plants, no wind) and `solve(net, enforce_q_lims)`. Voltages in p.u., the limits `V_MAX` = 1.05 and `V_MIN` = 0.95. In pandapower a shunt with **positive** `q_mvar` absorbs reactive power — a reactor.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)

# ---- the lecture's network and operating points ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
R_KM, X_KM, C_KM = 0.028, 0.32, 11.5
LOAD = {"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250}
PLANTS = {"Rijeka": 500, "Split": 600, "Osijek": 300}
V_MAX, V_MIN = 1.05, 0.95


def network(load_scale=1.0, plants=tuple(PLANTS), wind=600.0):
    net = pp.create_empty_network(sn_mva=100)
    bus = {name: pp.create_bus(net, 400, name=name) for name in BUSES}
    pp.create_ext_grid(net, bus["Zagreb"], vm_pu=1.0)
    for name in plants:
        pp.create_gen(net, bus[name], p_mw=PLANTS[name], vm_pu=1.0, name=f"plant {name}")
    pp.create_sgen(net, bus["Velebit"], p_mw=wind, q_mvar=0.0, name="wind Velebit")
    for name, p in LOAD.items():
        pp.create_load(net, bus[name], p_mw=p * load_scale, q_mvar=0.2 * p * load_scale, name=f"load {name}")
    for _, line in LINES.iterrows():
        pp.create_line_from_parameters(net, bus[line.bus0], bus[line.bus1], line.km, R_KM, X_KM, C_KM,
                                       max_i_ka=2.0, name=f"{line.bus0}-{line.bus1}")
    return net, bus


def night(load_scale=0.35):
    """The lecture's night: no plants online, no wind; load_scale as given."""
    return network(load_scale=load_scale, plants=(), wind=0.0)


def solve(net, enforce_q_lims=False):
    pp.runpp(net, numba=False, enforce_q_lims=enforce_q_lims)
    return net

### Warm-ups

**1. Surge impedance loading**

Write `sil(x_ohm_km, c_nf_km, kv)` returning the SIL in MW. Check it on the lecture's 400 kV line (537.6 MW) and compute it for a typical 220 kV line (0.40 Ω/km, 9 nF/km). How do the two compare with the power such lines usually carry?

In [ ]:
def sil(x_ohm_km: float, c_nf_km: float, kv: float) -> float:
    """Surge impedance loading (MW) of a lossless line."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.isclose(sil(X_KM, C_KM, 400), 537.6, atol=0.1)
print(f"400 kV: {sil(X_KM, C_KM, 400):.1f} MW; 220 kV: {sil(0.40, 9.0, 220):.1f} MW")

**2. How light is too light?**

With no plants online, the highest voltage at night depends on the load. Write `highest_voltage(load_scale)` and find, by bisection to 0.01, the load scale **below** which some voltage exceeds 1.05 p.u.

In [ ]:
def highest_voltage(load_scale: float) -> float:
    ...  # YOUR CODE HERE


def critical_load(low: float = 0.1, high: float = 1.5, tol: float = 0.01) -> float:
    """Smallest load scale at which every voltage is at or below V_MAX (no plants online)."""
    ...  # YOUR CODE HERE

In [ ]:
threshold = critical_load()
assert highest_voltage(threshold) <= V_MAX + 1e-9 < highest_voltage(threshold - 0.02)
print(f"below about {threshold:.2f} x peak load (about {threshold * sum(LOAD.values()):.0f} MW), "
      f"voltages exceed {V_MAX} p.u. with no plants online")

### Core

**3. A fixed reactor or a variable one?**

The lecture sized a 246 Mvar reactor at Split for one night (35 % load). For load scales [0.2, 0.35, 0.5, 0.7, 1.0, 1.3] (no plants online, no wind), report the highest and the lowest voltage (a) without a reactor, (b) with the **fixed** 246 Mvar reactor, and (c) the size a **variable** reactor at Split would need to keep every voltage ≤ 1.05 (0 if none is needed). Write `fixed_or_variable()`.

In [ ]:
def fixed_or_variable(scales=(0.2, 0.35, 0.5, 0.7, 1.0, 1.3), fixed_mvar: float = 246.0) -> pd.DataFrame:
    """Voltage range without and with a fixed reactor at Split, and the variable reactor size needed."""
    ...  # YOUR CODE HERE

In [ ]:
compare = fixed_or_variable()
assert compare.loc[0.35, "max_V_fixed"] <= V_MAX + 1e-3
assert compare.loc[1.3, "min_V_fixed"] < V_MIN, "the fixed reactor causes undervoltage at high load"
compare.round(3)

**4. A must-run plant instead of a reactor**

Instead of a reactor, keep the hydro plant at Split online at night at 100 MW, holding 1.0 p.u., with reactive limits ±Q. For Q in [100, 150, 200, 250, 300] Mvar, report the plant's reactive output and the highest voltage (use `enforce_q_lims=True`). Which capability is enough? Write `must_run(q_limits)`.

In [ ]:
def must_run(q_limits=(100, 150, 200, 250, 300)) -> pd.DataFrame:
    """Night (35 % load) with the Split hydro plant online at 100 MW and reactive limits +-Q."""
    ...  # YOUR CODE HERE

In [ ]:
plant = must_run()
assert plant.loc[300, "max_V"] <= V_MAX and plant.loc[100, "max_V"] > V_MAX
plant

**5. Interpretation**

What does task 2 say about how often this network needs help with voltage? Why does a fixed reactor sized for one night fail both on a lighter night and on a heavy day, and what does that imply for the kind of device to install? What does a must-run plant cost compared with a reactor — and what else does it bring?

*Your answer here.*

### Challenge

**6. Switch a line off?**

Operators sometimes open a lightly loaded line at night: an open line no longer produces reactive power. For the 35 % night without any compensation, open each line in turn and report the highest voltage, where it occurs, and the largest line loading. Write `switch_off()`.

In [ ]:
def switch_off() -> pd.DataFrame:
    """Night (35 % load, no compensation) with each line opened in turn."""
    ...  # YOUR CODE HERE

In [ ]:
switching = switch_off()
assert switching.max_V.astype(float).idxmin() == "open Osijek-Split"
switching

**7. Interpretation**

Opening a line removes its charging, yet most openings make the highest voltage **worse**. Why? Why is Osijek–Split the exception, and is opening it a good idea?

*Your answer here.*